In [ ]:
import pandas as pd

data = pd.read_csv('Unseen_Data.csv')

features_of_interest = ['family_history_diabetes', 'age', 'physical_activity_minutes_per_week', 'bmi', 'glucose_fasting']
features_to_scale = ['age', 'physical_activity_minutes_per_week', 'bmi', 'glucose_fasting']

y = data['diabetes_risk_score']
X = data[features_of_interest]

X.head()

In [ ]:
import numpy as np
import pickle

# load the imputer and scaler file
with open('scaler.pkl', 'rb') as file:
  loaded_scaler = pickle.load(file)

with open('imputer.pkl', 'rb') as file:
  loaded_imputer = pickle.load(file)



# Apply the initial data cleaning for 'age' using .loc on the DataFrame
X.loc[X["age"] == np.inf, "age"] = np.nan
X.loc[X["age"] <= 0, "age"] = np.nan
X.loc[(X["age"] % 1 != 0) & (~X["age"].isnull()), "age"] = np.nan


# Apply the imputation to the selected numerical features
X_imputed = loaded_imputer.transform(X)
X_imputed = pd.DataFrame(data=X_imputed, columns=features_of_interest)

X_imputed['physical_activity_minutes_per_week'] = np.cbrt(X_imputed['physical_activity_minutes_per_week'])

# Separate 'family_history_diabetes' from the features that need imputation and scaling
diabetes_history = X['family_history_diabetes']
X = X_imputed.drop(columns=['family_history_diabetes'], axis=1)

# # Apply the scaling to the imputed features
X_scaled = loaded_scaler.transform(X)
X_scaled = pd.DataFrame(data=X_scaled, columns=features_to_scale)

# Concatenate the original 'family_history_diabetes' with the scaled features
# The final X_test will be a DataFrame.
X_test = pd.concat([diabetes_history, X_scaled], axis = 1)
X_test.head()

In [ ]:
from tensorflow import keras

# Load the model
loaded_model = keras.models.load_model('saved_model.keras')

# Predict the diabeties score for the data
predictions = loaded_model.predict(X_test)

# Evaluate the predicted scores to the actual scores
test_loss, test_mae = loaded_model.evaluate(X_test, y)
print(f'Mean Absolute Error (MAE) on the test set: {test_mae:.4f}')

In [ ]:
# Print some of the predictions and the actual value
for actual, prediction in list(zip(y, predictions))[:5]:
  print(f'Actual: {actual}, Predicted: {prediction}')